In [0]:
from pyspark.sql.functions import *

kpi_total_revenue = fact_orders.agg(
    sum("revenue_usd").alias("total_revenue")
)

kpi_total_revenue.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.kpi_total_revenue")


kpi_revenue_country = fact_orders.groupBy("country_code") \
    .agg(sum("revenue_usd").alias("revenue"))

kpi_revenue_country.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.kpi_revenue_country")


kpi_revenue_channel = fact_orders.groupBy("channel") \
    .agg(sum("revenue_usd").alias("revenue"))

kpi_revenue_channel.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.kpi_revenue_channel")


kpi_completed_orders = fact_orders \
    .join(spark.table("workspace.gold.dim_status"), "status_id") \
    .filter(col("order_status") == "completed") \
    .agg(countDistinct("order_id").alias("completed_orders"))

kpi_completed_orders.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.kpi_completed_orders")


total_orders = fact_orders.select(countDistinct("order_id").alias("total")).collect()[0]["total"]

completed_orders = fact_orders \
    .join(spark.table("workspace.gold.dim_status"), "status_id") \
    .filter(col("order_status") == "completed") \
    .select(countDistinct("order_id").alias("completed")) \
    .collect()[0]["completed"]

kpi_completed_rate = spark.createDataFrame(
    [(completed_orders / total_orders,)],
    ["completed_order_rate"]
)

kpi_completed_rate.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.kpi_completed_order_rate")


kpi_aov = fact_orders.groupBy("order_id") \
    .agg(sum("revenue_usd").alias("order_value")) \
    .agg(avg("order_value").alias("AOV"))

kpi_aov.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.kpi_aov")


kpi_top_products = fact_orders \
    .groupBy("product_id") \
    .agg(sum("revenue_usd").alias("revenue")) \
    .orderBy(desc("revenue")) \
    .limit(5)

kpi_top_products.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.kpi_top_products")

kpi_active_customers = fact_orders \
    .agg(countDistinct("customer_id").alias("active_customers"))

kpi_active_customers.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.kpi_active_customers")

kpi_customer_acquisition = fact_orders \
    .withColumn("month", month("order_date")) \
    .groupBy("month") \
    .agg(countDistinct("customer_id").alias("new_customers"))

kpi_customer_acquisition.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.kpi_customer_acquisition")

silver_orders = spark.table("workspace.silver.silver_orders")

kpi_data_quality = silver_orders.agg(
    (sum(col("is_valid")) / count("*")).alias("data_quality_score")
)

kpi_data_quality.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.kpi_data_quality")